# Formaty plików do reprezentacji danych tekstowych

Na dzisiejszych zajęciach będziemy omawiać dwa typy formatów plików które będą przydatne w pracy z tekstem

Będziemy korzystali z biblioteki `pandas` która udostępnia bogaty zakres narzędzi do pracy z tabelami danych

https://pandas.pydata.org/docs/reference/index.html

In [ ]:
from pathlib import Path
import pandas
from tqdm import tqdm
from matplotlib import pyplot as plt

Pierwszym formatem z którym będziemy pracować, jest kolumnowy format `conllu`, służący do reprezentacji danych językowych zgodnie ze standardami **Universal Dependencies** (https://universaldependencies.org/introduction.html).

Pliki `conllu` zawierają listy zdań poddane procesowi wielowarstwowego opisu językoznawczego. Każdy z wierszy koresponduje z jednym tokenem, każda z kolumn koresponduje z jedną z warstw opisu. Kolumny są (w ramach wierszy) przedzielone znakiem tabulacji (`\t`), zaś tokeny przynależące do odrębnych zdań oddzielono podwójnym znakiem nowej linii (`\n`). Linie zawierające metadane poprzedzono znakiem `#`.

Korzystając z powyższej specyfikacji, moglibyśmy stosunkowo łatwo napisać własnoręcznie parser konwertujący te dane do obiektów Pythonowych, jednak wygodniej będzie skorzystać z gotowej biblioteki `conllu`:

In [ ]:
import conllu

DATA_DIR = Path("../../data")
UD_DATASET = "UD_Polish-PDB"

filepath = DATA_DIR / UD_DATASET / "pl_pdb-ud-train.conllu"
with open(filepath) as f:
    parsed = conllu.parse(f.read())

print(parsed[0])
print(type(parsed[0][0]), parsed[0][0], parsed[0][0].keys())

Parsowanie generuje listę obiektów klasy `TokenList`, z których każda jest iteratorem obiektów klasy `Token`, które zachowują się w sposób zbliżony do słowników Pythona. 

Wpierw utwórzmy tabelę (obiekt klasy `DataFrame`) składającą się z jednej kolumny: "tl", zawierającą wszystkie zdania z pliku. Zwróćmy uwagę, że każdy z wierszy jest automatycznie etykietowany indeksem - w tym wypadku liczbą porządkową, możemy je traktować jako numery zdań:

In [ ]:
sent_df = pandas.DataFrame({"tl": parsed})
sent_df

Pierwszą metodą obiektów klasy `DataFrame` jaką poznamy jest `explode`. Ma ona zastosowanie w momencie, gdy wartościami danej kolumny są iteratory. Tworzymy wówczas dłuższą tabelę, gdzie każdy wiersz odpowiada indywidualnemu elementowi z któregoś z iteratorów, możemy o tym myśleć jako o spłaszczeniu zagnieżdżonej listy list. Zwróćmy uwagę, że indeksy nie uległy zmianie, nowe wiersze odziedziczyły je po wierszach z których się wywodzą:

In [ ]:
token_dict_df = sent_df.explode("tl")
token_dict_df

Otrzymujemy w ten sposób tabelę, gdzie każdy z wierszy koresponduje z jednym tokenem.

Na tę chwilę, powyższa tabela nie różni się wiele od listy słowników. Taką strukturę, możemy łatwo przekształcić do nowej instancji `DataFrame`, gdzie każdemu z kluczy słownika zostanie przypisana odrębna kolumna:

In [ ]:
token_df = pandas.DataFrame.from_records(token_dict_df["tl"].to_list())
token_df

Indeksy nowej tabeli korespondują z numerami tokenów w pliku, natomiast warto zachować odniesienie do oryginalnych zdań. Ponieważ tabele zgadzają sie ze sobą wiersz-do-wiersza (albowiem drugą utworzono przez mapowanie pierwszej), możemy przypisać indeks starej, jako nową kolumnę nowej:

In [ ]:
token_df["sent_id"] = token_dict_df.index
token_df

Spróbujmy przygotować słownik wszystkich występujących w tekście słów. W tym celu wybierzemy kolumnę `lemma` odpowiadającą lematom - formom słownikowym (w szczególności nieodmiennym, oraz znormalizowanym co do sposobu zapisu).

Każda z kolumn jest obiektem klasy `Series`, który implementuje szereg interesujących metod. Wiele z tych metod (jak te z których skorzystamy) zwraca nowy obiekt klasy `Series` który to możemy poddawać dalszemu przetwarzaniu.

Skorzystamy z dwóch metod:
1. `drop_duplicates` - która domyślnie usuwa każde (poza pierwszym) wystąpienie wartości w kolumnie
2. `sort_values` - która posortuje wartości w kolumnie

In [ ]:
vocab = token_df["lemma"].drop_duplicates().sort_values()
vocab[1000:1050]

Jeżeli chcielibyśmy dowiedzieć się czegoś na temat tego, ile jest wystąpień każdej z unikalnych wartości w korpusie, możemy zastosować metodę `value_counts`:

In [ ]:
token_df["lemma"].value_counts()

## Ćwiczenie 1:

W kolumnie "upos" znajdują się tagi części mowy (Universal Part of Speech). Oblicz UDZIAŁ poszczególnych tagów w badanym korpusie.

In [ ]:
#TODO

### Ilustracja prawa Zipfa:

$word frequency \propto  \frac{1}{word ranking}$

Częstość występowania słowa jest proporcjonalna do odwrotności jego miejsca w rankingu.

Policzmy teoretyczną spodziewaną wartość na podstawie miejsca w rankingu, oraz obserwowaną częstość:

Na obiektach klasy `Series` możemy wykonywać zarówno skalarną, jak i wektorową arytmetykę.
`reset_index` przypisuje dla obiektu `DataFrame` indeksy na nowo, zachowując stare indeksowanie jako kolumnę, wykorzystujemy je tutaj by uzyskać wartości rankingu frekwencyjnego.

In [ ]:
zipf = token_df["lemma"].value_counts(normalize=True).reset_index()
max_prop = zipf["proportion"].max()
zipf["theoretical"] = max_prop / (zipf.index + 1)
zipf_cols = ["proportion", "theoretical"]
for col in zipf_cols:
    zipf[col].plot()


plt.legend(zipf_cols)
plt.xlim(0,100)
plt.show()

Metoda `apply` tworzy nową kolumnę, gdzie wartości są wynikiem zastosowania zadanej funkcji do kolumny wejściowej:

In [ ]:
token_df["charlen"] = token_df["form"].apply(len)
token_df[["form", "charlen"]]

Często będziemy korzystać z filtrowania tabel przez kolumnę wartości `bool`. Na przykład wybierzmy tylko te tokeny, które otagowano jako przysłówki, lub przymiotniki. Skorzystamy w tym celu z metody `isin` która sprawdza czy wartość w danym wierszu należy do podanego iteratora:

In [ ]:
adv_adj_filter = token_df["upos"].isin(["ADV", "ADJ"])
print(adv_adj_filter.iloc[:20])
adv_adj_df = token_df[adv_adj_filter]
adv_adj_df

W danych PDB mamy do czynienia z pewnego typu subtelnością - niektóre tokeny są oznaczone jako "subtokeny", wówczas w polu "id" nie mamy pojedynczego inta, lecz tuplę złożoną z dwóch takich intów. Dla prostoty analizy odfiltrujmy je, i utwórzmy nową tabelę.

Następnie wykorzystajmy metodę `apply` tym razem jednak nie jako metodę kolumny (`Series`) lecz całego `DataFrame`. Podając argument `axis=1`, specyfikujemy, że chcemy poruszać się po wierszach, mamy wówczas dostęp do wartości wszystkich z kolumn danego wiersza. 

Skorzystamy z dwóch takich kolumn: "id" oznaczającej numer tokenu w zdaniu, oraz "head" oznaczającej id bezpośredniego nadrzędnika dla danego tokenu (wartość 0, jeżeli token jest korzeniem zdania). Różnica między tymi dwoma oznacza "przesunięcie" krawędzi w drzewie zależnościowym, i.e. liczbę tokenów pomiędzy nadrzędnikiem, i danym podrzędnikiem (włącznie z tym ostatnim):

In [ ]:
atom_df = token_df[token_df["id"].apply(type) == int]
atom_df["dep_displacement"] = atom_df.apply(
    lambda row: row["head"] - row["id"] if row["head"] else 0, axis=1
)
atom_df[["id", "form", "head", "dep_displacement"]]

Teraz skorzystamy z funkcjonalności służących do agregacji danych:

Metoda `groupby` dzieli tabelę na podtabele, ze względu na wartość kolumny którą podajemy do jako argument. Zwraca ona iterator par: (wartość, podtabela), ale najczęściej nie iterujemy po nich bezpośrednio, lecz korzystamy z metod agregacji danych w ramach grup. Jedną z takich metod jest agregacja przez liczenie unikalnych wartości metodą `value_counts`:

Policzmy dystrybuantę długości (w znakach) słów z podziałem na klasy UPOS. W tym celu przygotowujemy:
1. Dane nt. wystąpień każdej z obserwowanych wartości "charlen", z podziałem na klasy.
2. Wybieramy każdą z interesujących nas klas metodą `loc` (pozwala ona na selekcję ze względu na indeks)
3. W ramach wyselekcjonowanych danych sortujemy je po indeksie (którym są teraz obserwowane wartości "charlen"), i stosujemy metodę `cumsum`, czyli sumę kumulatywną, aby uzyskać dystrybuantę.

In [ ]:
len_count = token_df.groupby("upos")["charlen"].value_counts()
print(len_count)
## cumulative distribution
len_freq = token_df.groupby("upos")["charlen"].value_counts(normalize=True)
POS = ["ADP", "VERB", "NOUN", "ADJ"]
for pos in POS:
    cdist = len_freq.loc[pos].sort_index().cumsum()
    cdist.plot()

plt.legend(POS)
plt.show()

## Ćwiczenie 2:

Policz średnią długość krawędzi ("dep_displacement") w każdej z grup wyodrębnionych ze względu na tag UPOS.

In [ ]:
#TODO

## Ćwiczenie 3:

Wybierz z całej tabeli te wiersze, które korespondują z tokenami, których lemat występuje min. 10 razy w całym korpusie.
Policz udział tagów UPOS w tak zawężonym zbiorze, a następnie porównaj go (i.e. przedstaw stosunek, korzystając z arytmetyki wektorów) z udziałem w całym korpusie.

In [ ]:
#TODO

Bardzo istotną metodą pracy z tabelami, jest możliwość łączenia ich, w oparciu o znajdowane w nich wartości.

Najczęściej wartościami takimi są różnego rodzaju identyfikatory, wyobraźmy sobie np. że mamy dwie tabele dot. produktów w sklepie: w pierwszej znajdują się dane nt. samych produktów (np. ich cena, masa), w drugiej dane nt. producentów (np. kraj siedziby). W każdej z tych tabel jest także kolumna zawierająca identyfikator producenta. Możemy połączyć dwie tabele **po** kluczu identyfikatora, aby w jednej tabeli mieć sparowane dane z obu źródeł, i.e. będziemy w stanie np. dowiedzieć się z jakiego kraju pochodzi dany produkt.

W `pandas` jedną z metod która umożliwia takie operacje jest `merge`. W naszym wypadku będziemy łączyli tabelę samą ze sobą, jednak połączenie będzie się odbywało między różnymi wierszami. Kluczem po którym będziemy łączyli tabele, będzie para `("sent_id", "head")` dla lewej tabeli, i `("sent_id", "id")`. Oznacza to, że dla każdego wiersza z lewej tabeli, będziemy znajdowali wiersz z prawej tabeli taki, że wartości "sent_id" są identyczne w obu, zaś wartość "head" z lewego wiersza jest tożsama z wartością "id" w prawym.

Albo mówiąc po prostu, połączymy podrzędniki z ich nadrzędnikami.

dodatkowo podajemy wartość dla parametru `suffixes`, która określi jak przemianować identycznie brzmiące klucze z lewej i prawej tabeli:

In [ ]:
dep_df = atom_df.merge(
    atom_df,
    left_on=("sent_id", "head"),
    right_on=("sent_id", "id"),
    suffixes=("_child", "_head"),
)
dep_df

W ramach wyodrębnionych grup, możemy przeprowadzać agregację wartości kolumn, korzystamy w tym celu z metody `agg` do której podajemy słownik o strukturze `key: agg_name` gdzie `key` to nazwa kolumny, zaś `agg_name` nazwa funkcji agregującej (np. `mean`, `max`, `var` etc.).

Podzielmy krawędzie ze względu na etykiety UPOS obu ich członów, i obliczmy wariancję przesunięcia względem nadrzędnika: 

In [ ]:
dep_df.groupby(["upos_child", "upos_head"]).agg({"dep_displacement_child": "var"})

Możemy wykorzystać różne funkcje agregujące i kolumny na raz:

In [ ]:
token_df.groupby("upos").agg({"charlen": "mean", "deprel": "nunique", "id": "count"})

Jedną z funkcji agregujących jest `list` która pozwala zgrupować wartości w jeden iterator, możemy w ten sposób zgrupować podrzędniki każdego z tokenów:

In [ ]:
child_lists = dep_df.groupby(["id_head", "sent_id"]).agg({"form_head": "first", "form_child": list})
child_lists

## Format JSON

Innym ważnym do poznania formatem do pracy z danymi tekstowymi jest JSON. Pozwala on na reprezentację dowolnie zagnieżdżonych struktur o charakterze zbliżonym do Pythonowych słowników. `pandas` pozwala również na pracę z takimi plikami:

In [ ]:
SQUAD_DATASET = "SQuAD2.0"
filepath = DATA_DIR / SQUAD_DATASET / "dev-v2.0.json"
squad_df = pandas.read_json(filepath)
squad_df

Interesuje nas tylko kolumna "data" która sama ma strukturę słownika:

In [ ]:
squad_df = squad_df["data"].apply(pandas.Series)
squad_df

Każdy z wierszy odpowiada artykułowi, który jest powiązany z listą paragrafów. Zastosujmy metodę `explode`:

In [ ]:
par_df = squad_df.explode("paragraphs")
par_df

Jeszcze głębiej znajdują się klucze "qas" (zawierający listę pytań) i "context" zawierający tekst, do którego pytania te ułożono. Przepiszmy te wartości jako odrębne kolumny korzystając z metody `apply` której podamy anonimową funkcję mającą za zadanie jedynie odpakowanie odpowiednich kluczy:

In [ ]:
par_df["context"] = par_df.apply(lambda x: x["paragraphs"]["context"], axis=1)
par_df["qas"] = par_df.apply(lambda x: x["paragraphs"]["qas"], axis=1)
par_df

Możemy wreszcie wykorzystać `explode` aby uzyskać strukturę, gdzie każdy wiersz będzie zawierał tylko jedno pytanie:

In [ ]:
raw_q_df = par_df.explode("qas")[["title", "qas", "context"]]
raw_q_df

Ponownie, dane w kolumnie "qas" są zagnieżdżone w słownikach. Aby je odpakować, najwygodniej będzie tym razem zastosować konkatenację reszty oryginalnej tabeli, z przetworzonymi wartościami wspomnianej kolumny:

In [ ]:
q_df = pandas.concat([raw_q_df[["title", "context"]], raw_q_df["qas"].apply(pandas.Series)], axis=1)
q_df

## Ćwiczenie 4

Dla każdego pytania, w kolumnie "answers" znajduje się iterator zawierający wiele wartości (będących słownikami), każda z odpowiedzi pochodzi od innego anotatora. Jeżeli odpowiedzi są identyczne, oznacza to zgodę anotatorów, jeżeli różnią się, oznacza różnicę zdań.

Wyodrębnij podzbiór tabeli `q_df` zdefiniowany przez pytania, dla których anotatorzy osiągnęli pełną zgodę co do odpowiedzi

In [ ]:
#TODO
#1. wydobądź odpowiedzi w sposób który pozwala na ich indywidualną analizę, jednocześnie zachowując odniesienie do pytań
#2. znajdź sposób na porównanie ze sobą odpowiedzi
#3. dla każdego pytania policz liczbę unikalnych odpowiedzi
#4. przefiltruj tabelę pytań korzystając z liczby odpowiedzi